# Devoir 5 — Agent IA : Assistant de recettes

L’objectif de ce notebook est de montrer le fonctionnement général du projet, les choix techniques, les outils développés et la manière dont l’agent est testé.

Le cas d’usage choisi est un **assistant de recettes**. L’utilisateur peut demander à l’agent de consulter des recettes, de chercher une recette à partir d’un ingrédient, de proposer une recette selon ce qu’il a chez lui ou encore de générer une liste de courses.

## 1. Objectif du projet

L’objectif n’est pas de créer un simple chatbot qui répond uniquement avec du texte. Le but est de créer un agent capable d’utiliser des outils Python pour agir sur des données locales.

Dans ce projet, les recettes sont stockées dans le fichier `recipes.json`. L’agent peut consulter ou modifier ce fichier grâce à des fonctions Python définies dans `recipe_agent/tools.py`.

L’utilisateur n’appelle pas directement ces fonctions. Il formule une demande en langage naturel, par exemple :

> J’ai des œufs, du fromage et du beurre. Que peux-tu me proposer ?

L’agent doit alors comprendre la demande et utiliser l’outil adapté.

## 2. Lien avec l’atelier de M. Suire

Le projet reprend les principes vus pendant l’atelier sur les agents IA :

- utilisation de **Google ADK** pour définir l’agent ;
- utilisation de **LiteLLM** pour relier l’agent au modèle de langage ;
- utilisation d’**Ollama** pour exécuter un modèle localement ;
- création d’**outils Python** que l’agent peut appeler ;
- stockage des données dans un fichier **JSON** ;
- tests unitaires avec **pytest** ;
- observation du comportement de l’agent dans **ADK Web**.

## 3. Structure du projet

Le projet est organisé de manière à séparer clairement les responsabilités.

```text
agent-ia-recettes/
│
├── recipe_agent/
│   ├── __init__.py
│   ├── agent.py          # Définition de l’agent ADK
│   └── tools.py          # Outils Python utilisés par l’agent
│
├── tests/
│   └── test_tools.py     # Tests unitaires des outils
│
├── use_case/
│   ├── use_case_agent_recettes.puml
│   └── use_case_agent_recettes.png
│
├── recipes.json          # Données persistantes
├── pyproject.toml        # Dépendances et configuration du projet
├── README.md
├── uv.lock
└── agent_ia_assistant_recettes_NOTEBOOK.ipynb
```

Le fichier le plus important pour les actions concrètes est `tools.py`. Le fichier `agent.py` sert à déclarer l’agent, son prompt système, son modèle et les outils qu’il peut utiliser.

## 4. Différence entre chatbot et agent IA

Un chatbot classique produit principalement une réponse textuelle. Il peut expliquer une recette ou donner une idée, mais il n’agit pas directement sur les données du projet.

Un agent IA peut aller plus loin : il peut choisir et exécuter des outils.

Dans ce projet :

```text
Utilisateur → message en langage naturel → agent ADK → modèle de langage → choix d’un outil Python → lecture ou modification de recipes.json → réponse finale
```

Le modèle de langage ne modifie donc pas directement le fichier JSON. Il sert à interpréter la demande et à choisir l’action adaptée. L’action réelle est effectuée par une fonction Python.

## 5. Technologies utilisées

| Élément | Rôle dans le projet |
|---|---|
| Google ADK | Définit et orchestre l’agent IA |
| LiteLLM | Connecte l’agent au modèle de langage |
| Ollama | Exécute localement le modèle LLM |
| Python | Implémente les outils appelables par l’agent |
| JSON | Stocke les recettes de manière persistante |
| pytest | Vérifie les outils Python de façon automatique |

Cette organisation permet de séparer la partie non déterministe, liée au modèle de langage, de la partie déterministe, liée aux fonctions Python.

## 6. Présentation des outils Python

Les outils sont des fonctions Python classiques. Ils sont définis dans `recipe_agent/tools.py` et sont rendus accessibles à l’agent dans `recipe_agent/agent.py`.

| Outil | Rôle |
|---|---|
| `add_recipe` | Ajoute une recette dans `recipes.json` |
| `list_recipes` | Liste les recettes enregistrées |
| `search_recipes_by_ingredient` | Recherche les recettes contenant un ingrédient |
| `suggest_recipe` | Propose une recette selon les ingrédients disponibles |
| `generate_shopping_list` | Génère une liste de courses pour une recette |
| `delete_recipe` | Supprime une recette à partir de son identifiant |

Dans ce notebook, les démonstrations utilisent surtout les outils de consultation afin de ne pas modifier inutilement les données de présentation.

## 7. Chargement des outils

Le notebook étant placé à la racine du projet, les imports restent simples. Cette cellule charge les outils Python qui seront utilisés dans les démonstrations.

In [ ]:
from recipe_agent.tools import (
    add_recipe,
    delete_recipe,
    delete_recipe_by_name,
    generate_shopping_list,
    list_recipes,
    search_recipes_by_ingredient,
    suggest_recipe,
)

## 8. Données de départ

Les recettes sont stockées dans le fichier `recipes.json`. Ce fichier contient quelques recettes de base utilisées pour tester le fonctionnement de l’agent.

Le fait d’utiliser un fichier JSON permet d’avoir une persistance simple : les données restent présentes même après l’arrêt du notebook ou d’ADK Web.

In [2]:
print(list_recipes())

[1] Pâtes tomate basilic - plat - facile - 25 min | ingrédients : pâtes, tomate, basilic, huile d'olive, ail
[2] Omelette fromage - plat - facile - 10 min | ingrédients : oeufs, fromage, sel, poivre, beurre
[3] Salade de pois chiches - entree - facile - 15 min | ingrédients : pois chiches, concombre, tomate, citron, persil, huile d'olive
[4] Crêpes simples - dessert - facile - 30 min | ingrédients : farine, oeufs, lait, sucre, beurre
[5] Wrap poulet curry - plat - facile - 15 min | ingrédients : tortilla, poulet, yaourt nature, curry, salade, concombre, sel, poivre
[6] Soupe de légumes - entree - facile - 25 min | ingrédients : carotte, pomme de terre, poireau, oignon


## 9. Recherche par ingrédient

Le premier outil testé est `search_recipes_by_ingredient`. Il permet de retrouver les recettes qui contiennent un ingrédient donné.

Ce test correspond à une demande utilisateur du type :

> Quelles recettes contiennent de la tomate ?

In [3]:
print(search_recipes_by_ingredient("tomate"))

Recettes contenant 'tomate' :
- [1] Pâtes tomate basilic (25 min, facile)
- [3] Salade de pois chiches (15 min, facile)


## 10. Suggestion de recette

L’outil `suggest_recipe` propose une recette en fonction des ingrédients disponibles chez l’utilisateur.

Dans l’exemple suivant, l’utilisateur possède des œufs, du fromage et du beurre. L’outil recherche la recette la plus adaptée parmi les recettes enregistrées.

In [4]:
print(suggest_recipe(
    available_ingredients=["oeufs", "fromage", "beurre"],
    max_time_minutes=15,
))

Suggestion : Omelette fromage (id: 2).
Ingrédients disponibles utilisés : beurre, fromage, oeufs.
Ingrédients manquants : poivre, sel.
Temps estimé : 10 min. Difficulté : facile.


## 11. Génération d’une liste de courses

L’outil `generate_shopping_list` compare les ingrédients nécessaires à une recette avec les ingrédients déjà disponibles.

Dans cet exemple, l’utilisateur veut préparer la recette numéro 1 et possède déjà des pâtes et de la tomate. L’outil retourne uniquement les ingrédients manquants.

In [6]:
print(generate_shopping_list(recipe_id=1, available_ingredients=["pâtes", "tomate"],))

print(generate_shopping_list(recipe_name="Crêpes simples"))

Liste de courses pour 'Pâtes tomate basilic' :
- ail
- basilic
- huile d'olive
Liste de courses pour 'Crêpes simples' :
- beurre
- farine
- lait
- oeufs
- sucre


## 12. Ajout et suppression de recettes

Les outils `add_recipe`, `delete_recipe` et `delete_recipe_by_name` permettent de modifier réellement le fichier `recipes.json`.

Pour éviter de modifier les données principales pendant la présentation du notebook, ces outils ne sont pas exécutés directement dans cette section. Leur fonctionnement est toutefois vérifié automatiquement dans `tests/test_tools.py`, à l’aide d’un fichier JSON temporaire. Cela permet de tester l’ajout et la suppression de recettes sans altérer la base principale du projet.

Exemple de demande d’ajout à tester dans ADK Web :

```text
Ajoute une recette de soupe de légumes avec carotte, pomme de terre, poireau et oignon.

## 13. Définition de l’agent ADK

L’agent est défini dans le fichier `recipe_agent/agent.py`.

Ce fichier contient :

- le nom de l’agent ;
- le modèle utilisé via LiteLLM ;
- le prompt système ;
- la liste des outils Python accessibles à l’agent ;
- l’objet `root_agent`, utilisé par Google ADK.

L’agent n’est pas lancé directement depuis ce notebook.

Depuis la racine du dépôt, il peut être lancé avec la commande suivante :

```bash
uv --directory "part5 - agent_ia_recettes" run adk web
```

Si le terminal est déjà placé dans le dossier `part5 - agent_ia_recettes`, la commande est :

```bash
uv run adk web
```

Une fois l’interface ouverte, il faut sélectionner l’agent `recipe_agent`.

## 14. Prompt système

Le prompt système définit le comportement attendu de l’agent pendant la conversation.

Il précise notamment que l’agent doit :

- rester dans le domaine des recettes et de la cuisine ;
- utiliser les outils Python disponibles pour ajouter, rechercher, lister ou supprimer des recettes ;
- consulter les données existantes lorsqu’un utilisateur demande une recette déjà présente, au lieu d’inventer une réponse ;
- vérifier avec les outils qu’une recette portant le même nom n’existe pas déjà avant d’en ajouter une nouvelle ;
- confirmer brièvement lorsqu’une recette a bien été ajoutée avec `add_recipe` ;
- proposer ensuite 1 ou 2 idées de recettes similaires, de variantes ou d’accompagnements ;
- ne jamais ajouter automatiquement ces suggestions sans confirmation de l’utilisateur ;
- refuser poliment les demandes qui ne concernent pas les recettes.

Ce prompt est important parce qu’un modèle de langage peut parfois répondre de manière générale ou inventer une information. Les consignes permettent donc de guider l’agent pour qu’il utilise les outils lorsque c’est nécessaire, qu’il évite les doublons et qu’il reste dans le cadre du projet.

## 15. Suggestions automatiques via le LLM

Dans l’atelier de l’intervenant, les suggestions automatiques ne sont pas implémentées comme un nouvel outil Python déterministe. Elles sont pilotées par le **prompt système** et générées par le **LLM** à partir du contexte.

Dans ce projet, cette idée est adaptée au domaine des recettes : après l’ajout réussi d’une recette, l’agent confirme l’ajout puis propose 1 ou 2 idées de recettes similaires, de variantes ou d’accompagnements. Ces propositions ne sont pas ajoutées automatiquement dans `recipes.json` : l’utilisateur doit confirmer s’il veut réellement les ajouter.

Cette partie montre une différence importante entre deux types de comportements :

- les outils comme `list_recipes`, `search_recipes_by_ingredient` ou `generate_shopping_list` sont **déterministes** : à données identiques, ils donnent le même résultat ;
- les suggestions automatiques sont **non déterministes** : elles dépendent du modèle, du contexte et de la formulation de la demande.

Exemple de test dans ADK Web :

```text
Utilisateur : Ajoute une recette appelée "Wrap poulet curry" avec tortilla, poulet, yaourt nature, curry, salade et concombre. La recette prend 15 minutes et elle est facile.

Comportement attendu :
1. l’agent appelle l’outil add_recipe ;
2. il confirme que la recette a été ajoutée ;
3. il propose ensuite 1 ou 2 idées proches, par exemple une variante végétarienne ou une sauce d’accompagnement ;
4. il n’ajoute pas ces suggestions sans confirmation.
```

Ce comportement repose sur la capacité du LLM à générer des idées pertinentes. Il est donc utile pour enrichir l’expérience utilisateur, mais il doit rester encadré par le prompt système afin d’éviter que l’agent modifie les données sans accord explicite.


## 16. Fonctionnement attendu dans ADK Web

Le fonctionnement attendu suit la logique ReAct vue en atelier :

1. l’utilisateur envoie une demande ;
2. le modèle interprète cette demande ;
3. l’agent choisit un outil ;
4. l’outil Python est exécuté ;
5. le résultat de l’outil est renvoyé à l’agent ;
6. l’agent formule une réponse finale.

Exemple :

```text
Utilisateur : J’ai des œufs, du fromage et du beurre. Que peux-tu me proposer ?

Outil attendu : suggest_recipe

Résultat attendu : l’agent propose une recette compatible avec les ingrédients disponibles.
```

Cette étape permet de vérifier que l’agent ne se contente pas de générer du texte, mais qu’il utilise réellement les outils disponibles.

## 17. Diagramme use case

Le diagramme use case présente les principales interactions entre l’utilisateur et l’agent.

Il montre aussi que certaines actions consultent les données stockées, tandis que d’autres les modifient.

## 18. Organisation des tests

Les tests sont organisés à deux niveaux.

### Tests unitaires des outils

Les tests unitaires se trouvent dans le fichier :

```text
tests/test_tools.py
```

Ils vérifient les fonctions Python indépendamment du modèle de langage. Ces tests sont déterministes : une même entrée doit produire le même résultat.

Ils vérifient notamment :

- l’ajout d’une recette ;
- le refus d’une difficulté invalide ;
- le refus d’un doublon ;
- la recherche par ingrédient ;
- la suggestion de recette ;
- la génération d’une liste de courses ;
- la suppression d’une recette.

Commande utilisée :

```bash
uv --directory "part5 - agent_ia_recettes" run pytest tests/ -v
```

Si le terminal est déjà placé dans le dossier `part5 - agent_ia_recettes/` :

```bash
uv run pytest tests/ -v
```

### Tests manuels dans ADK Web

Les tests manuels consistent à envoyer des messages à l’agent dans l’interface ADK Web. Ils permettent de vérifier que l’agent comprend une demande en langage naturel et choisit le bon outil.

## 19. Scénarios de tests manuels

| Scénario | Message envoyé à l’agent | Résultat attendu |
|---|---|---|
| Afficher les recettes | `Liste les recettes disponibles.` | L’agent affiche les recettes de `recipes.json`. |
| Rechercher par ingrédient | `Quelles recettes contiennent de la tomate ?` | L’agent retourne les recettes contenant de la tomate. |
| Suggérer une recette | `J’ai des œufs, du fromage et du beurre. Que peux-tu me proposer ?` | L’agent utilise `suggest_recipe`. |
| Générer une liste de courses | `Que dois-je acheter pour faire les pâtes tomate basilic si j’ai déjà des pâtes et de la tomate ?` | L’agent utilise `generate_shopping_list`. |
| Ajouter une recette | `Ajoute une recette de soupe de légumes avec carotte, pomme de terre, poireau et oignon.` | L’agent utilise `add_recipe`. |
| Suggestion automatique via LLM | `Ajoute une recette appelée Wrap poulet curry avec tortilla, poulet, yaourt nature, curry, salade et concombre.` | L’agent utilise `add_recipe`, confirme l’ajout puis propose 1 ou 2 idées de recettes similaires sans les ajouter automatiquement. |
| Supprimer une recette | `Supprime la recette avec l’identifiant 4.` | L’agent utilise `delete_recipe`. |
| Demande hors périmètre | `Quel temps fera-t-il demain ?` | L’agent refuse poliment. |

Ces tests complètent les tests unitaires. Les tests unitaires valident les outils, tandis que les tests dans ADK Web valident le comportement global de l’agent.

## 20. Limites observées

L’agent reste volontairement basique. Le fichier JSON est suffisant pour un petit projet, mais il serait limité pour une application plus complète.

Les ingrédients sont stockés sous forme de texte simple. La liste de courses indique donc les ingrédients manquants, mais pas les quantités exactes.

Le comportement de l’agent dépend aussi du modèle utilisé. Avec un petit modèle local, certaines formulations peuvent être mal interprétées. L’agent peut parfois hésiter entre plusieurs outils ou demander une précision. C’est pour cela que le prompt système doit être clair et que les outils doivent gérer les erreurs.

Enfin, les tests du comportement global restent manuels. Ils permettent de vérifier l’agent dans ADK Web, mais ils ne remplacent pas une suite complète de tests automatisés sur les échanges avec le modèle.

## 21. Améliorations possibles

Plusieurs améliorations pourraient être ajoutées :

- ajouter des quantités aux ingrédients ;
- ajouter des tags comme végétarien, rapide, économique ou sans gluten ;
- utiliser SQLite au lieu d’un fichier JSON ;
- ajouter une confirmation avant suppression ;
- connecter l’agent à une API de recettes.

Ces améliorations permettraient de rendre l’agent plus robuste et plus proche d’une application réelle.

## 22. Conclusion

Ce projet m’a permis de mettre en pratique les principes vus pendant l’atelier de Mr Suire sur les agents IA. L’agent assistant de recettes montre qu’un modèle de langage ne sert pas uniquement à produire une réponse textuelle. Il peut aussi être intégré dans une architecture où il choisit et déclenche des outils Python.

Le projet reprend les éléments importants de l’atelier : Google ADK pour définir l’agent, LiteLLM pour connecter le modèle, Ollama pour exécuter un modèle local, des outils Python pour réaliser les actions, un fichier JSON pour conserver les données et des tests pour valider les fonctions.

Le choix du cas d’usage des recettes permet d’adapter cette architecture à un domaine différent de l’exemple de gestion de tâches. L’agent peut consulter une base locale, rechercher des recettes, proposer un plat selon les ingrédients disponibles, générer une liste de courses et proposer automatiquement des idées de recettes similaires grâce au LLM.

La réalisation du projet montre aussi qu’un agent IA doit être conçu comme un petit système logiciel. Il ne suffit pas d’écrire un prompt : il faut définir un périmètre, organiser les outils, gérer les erreurs, tester les fonctions et observer le comportement réel de l’agent. Même si l’agent reste simple, il constitue une base claire pour comprendre comment un LLM peut être utilisé avec des outils pour agir sur des données locales.